# Export Qwen3.5-4B to ExecuTorch .pte Format

Exports Qwen3.5-4B using the ExecuTorch `export_llm` pipeline on Colab.

Notes:
- Current path is fp32/static-shape (large files).
- Expect very large artifacts for 4B fp32 (roughly 15-18GB).
- For app deployment, quantization is still needed for practical size.


In [ ]:
# Optional: mount Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
%%bash
set -e
cd /content
if [ ! -d executorch ]; then
  git clone https://github.com/Phineas1500/executorch.git
fi
cd /content/executorch
git fetch --all
git checkout qwen3_5_phase2
git pull --ff-only origin qwen3_5_phase2
git submodule sync --recursive
git submodule update --init --recursive


In [ ]:
%%bash
set -e
cd /content/executorch
python -m pip install -U pip
./install_executorch.sh --editable
python -m pip install safetensors ruamel.yaml tabulate pandas huggingface_hub


In [ ]:
%%bash
set -e
cd /content/executorch
cmake -S third-party/flatbuffers -B /tmp/flatbuffers-build -DFLATBUFFERS_BUILD_FLATC=ON -DFLATBUFFERS_BUILD_TESTS=OFF -DFLATBUFFERS_BUILD_FLATHASH=OFF
cmake --build /tmp/flatbuffers-build -j


In [ ]:
# Optional: HF login (skip if public download works without auth)
from huggingface_hub import login
login()


In [ ]:
%%bash
set -e
cd /content/executorch
OMP_NUM_THREADS=1 TORCHINDUCTOR_COMPILE_THREADS=1 \
PATH=/tmp/flatbuffers-build:$PATH \
python -m extension.llm.export.export_llm \
  --config examples/models/qwen3_5/config/qwen3_5_xnnpack_fp32.yaml \
  +base.model_class=qwen3_5_4b \
  +base.params=examples/models/qwen3_5/config/4b_config.json \
  backend.xnnpack.enabled=False \
  export.max_seq_length=1 \
  export.max_context_length=1 \
  +export.output_name=/content/qwen3_5_4b_no_backend_smoke.pte


In [ ]:
%%bash
set -e
cd /content/executorch
OMP_NUM_THREADS=1 TORCHINDUCTOR_COMPILE_THREADS=1 \
PATH=/tmp/flatbuffers-build:$PATH \
python -m extension.llm.export.export_llm \
  --config examples/models/qwen3_5/config/qwen3_5_xnnpack_fp32.yaml \
  +base.model_class=qwen3_5_4b \
  +base.params=examples/models/qwen3_5/config/4b_config.json \
  export.max_seq_length=128 \
  export.max_context_length=128 \
  +export.output_name=/content/qwen3_5_4b_xnnpack_fp32_128.pte


In [ ]:
%%bash
set -e
ls -lh /content/qwen3_5_4b_*.pte


## Download Options

For very large files, direct browser download may fail.
Use chunked download or copy to Drive.


In [ ]:
%%bash
set -e
split -b 1024m /content/qwen3_5_4b_xnnpack_fp32_128.pte /content/qwen3_5_4b_xnnpack_fp32_128.pte.part.
ls -lh /content/qwen3_5_4b_xnnpack_fp32_128.pte.part.*


In [ ]:
import glob
from google.colab import files
for path in sorted(glob.glob('/content/qwen3_5_4b_xnnpack_fp32_128.pte.part.*')):
    files.download(path)
